# Escenario 5 — Claude Code para Integración Continua

**Dominios:** Claude Code Configuration & Workflows · Prompt Engineering & Structured Output

## Contexto
Integrás Claude Code en tu pipeline CI/CD para revisiones de código automatizadas, generación de tests y feedback en pull requests. El desafío: feedback accionable que minimice falsos positivos.

## Lo que vas a practicar
1. Salida estructurada con tool_use y JSON schema para CI
2. Criterios explícitos de revisión (reducción de falsos positivos)
3. Few-shot examples para código ambiguo
4. Revisión multi-pase para PRs grandes
5. Message Batches API para cargas no bloqueantes
6. Evitar comentarios duplicados en sucesivos commits

In [ ]:
import anthropic
import json
from typing import Optional

client = anthropic.Anthropic()

## Paso 1 — Tool de revisión con JSON schema

**Concepto clave:** `tool_use` con JSON schema garantiza salida estructurada (sin errores de sintaxis JSON). Forzar la tool con `tool_choice` asegura que siempre se ejecute.

In [ ]:
REVIEW_TOOL = {
    "name": "report_code_review",
    "description": "Reporta los hallazgos de la revisión de código en formato estructurado",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file": {"type": "string"},
                        "line": {"type": ["integer", "null"]},
                        "severity": {
                            "type": "string",
                            "enum": ["HIGH", "MEDIUM", "LOW"]
                        },
                        "category": {
                            "type": "string",
                            "enum": ["security", "bug", "performance", "logic"]
                        },
                        "description": {"type": "string"},
                        "suggested_fix": {"type": ["string", "null"]},
                        "detected_pattern": {
                            "type": "string",
                            "description": "Patrón de código detectado — para análisis de falsos positivos"
                        }
                    },
                    "required": ["file", "severity", "category", "description", "detected_pattern"]
                }
            },
            "summary": {"type": "string"},
            "files_reviewed": {"type": "integer"}
        },
        "required": ["findings", "summary", "files_reviewed"]
    }
}

## Paso 2 — System prompt con criterios explícitos y few-shot

In [ ]:
REVIEW_SYSTEM_PROMPT = """
Sos un revisor de código automatizado para un pipeline CI.

REPORTAR SOLO:
- Bugs: comportamiento que claramente contradice el intent del código
- Seguridad: SQL injection, XSS, secrets expuestos, deserialización insegura

NO REPORTAR:
- Estilo (nombres de variables, formato, longitud de línea)
- Preferencias personales
- Mejoras opcionales de rendimiento sin impacto medible
- Patrones alternativos igual de correctos

CRITERIOS DE SEVERIDAD:
- HIGH: comportamiento incorrecto en producción con datos reales
  Ejemplo: off-by-one que silencia errores, SQL injection, auth bypass
- MEDIUM: falla bajo condiciones específicas (carga alta, datos edge)
  Ejemplo: race condition, conexión no cerrada en excepción
- LOW: funciona pero con riesgo latente
  Ejemplo: deprecated API que se eliminará en v4.0

EJEMPLOS DE DECISIÓN:

Código:
  query = f"SELECT * FROM users WHERE id = '{user_id}'"
Decisión: REPORTAR — SQL injection, HIGH, security
Razón: user_id no sanitizado puede contener SQL malicioso

Código:
  # Usar snake_case en lugar de camelCase
  def getUserById(id):
Decisión: NO REPORTAR — es estilo, no un bug

Código:
  conn = db.connect()
  result = conn.query(sql)
  conn.close()  # Si query() lanza, esto nunca se ejecuta
Decisión: REPORTAR — MEDIUM, bug, resource leak en excepción
"""

In [ ]:
def review_code(diff_content: str, previous_findings: Optional[list] = None) -> dict:
    """Revisa código y devuelve hallazgos estructurados. Evita duplicar hallazgos previos."""
    
    user_message = f"Revisá este diff:\n\n{diff_content}"
    
    if previous_findings:
        prev_summary = json.dumps(previous_findings, indent=2, ensure_ascii=False)
        user_message += f"\n\nHALLAZGOS PREVIOS (NO reportar estos nuevamente):\n{prev_summary}"
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=REVIEW_SYSTEM_PROMPT,
        tools=[REVIEW_TOOL],
        tool_choice={"type": "tool", "name": "report_code_review"},
        messages=[{"role": "user", "content": user_message}]
    )
    
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    
    return {"findings": [], "summary": "No findings", "files_reviewed": 0}

In [ ]:
# Diff con problemas reales para revisar
test_diff = """
diff --git a/src/api/users.py b/src/api/users.py
@@ -40,15 +40,20 @@ class UserRouter:
 
     async def get_user(self, user_id: str):
-        user = await self.db.users.find_one({"id": user_id})
+        # Optimización: usar query directa
+        query = f\"SELECT * FROM users WHERE id = '{user_id}'\"
+        user = await self.db.execute(query)
         if not user:
             raise HTTPException(status_code=404)
         return user
 
     async def update_user_email(self, user_id: str, new_email: str):
-        await self.db.users.update({"id": user_id}, {"email": new_email})
+        conn = await self.db.connect()
+        await conn.execute(f\"UPDATE users SET email='{new_email}' WHERE id='{user_id}'\")
+        await conn.close()
         return {"status": "updated"}
"""

print("Ejecutando revisión de código...")
result = review_code(test_diff)
print(json.dumps(result, indent=2, ensure_ascii=False))

## Paso 3 — Revisión multi-pase para PRs grandes

In [ ]:
def review_large_pr(files: dict[str, str]) -> dict:
    """
    Revisión multi-pase para PRs con muchos archivos.
    Pase 1: análisis local por archivo (profundidad consistente)
    Pase 2: análisis de integración cross-file (flujo de datos)
    """
    
    print(f"PR con {len(files)} archivos → usando revisión multi-pase")
    print()
    
    # PASE 1: Analizar cada archivo individualmente
    print("PASE 1: Análisis local por archivo")
    local_findings = {}
    
    for filename, content in files.items():
        print(f"  → Analizando {filename}...")
        result = review_code(f"Archivo: {filename}\n\n{content}")
        local_findings[filename] = result.get("findings", [])
        print(f"     {len(local_findings[filename])} issues encontrados")
    
    # PASE 2: Análisis de integración cross-file
    print()
    print("PASE 2: Análisis de integración cross-file")
    
    files_summary = "\n".join([
        f"Archivo: {name}\nIssues locales: {len(findings)}\n" 
        for name, findings in local_findings.items()
    ])
    
    integration_response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=REVIEW_SYSTEM_PROMPT,
        tools=[REVIEW_TOOL],
        tool_choice={"type": "tool", "name": "report_code_review"},
        messages=[{"role": "user", "content": f"""
Analizá issues de INTEGRACIÓN entre estos archivos. 
Buscá: flujo de datos incorrecto entre módulos, inconsistencias de contrato,
dependencias circulares, estado compartido mal gestionado.
NO repetir issues locales ya identificados.

{files_summary}

Contenido de los archivos:
{'\n---\n'.join([f'{name}:\n{content}' for name, content in files.items()])}
"""}]
    )
    
    integration_findings = []
    for block in integration_response.content:
        if block.type == "tool_use":
            integration_findings = block.input.get("findings", [])
    
    print(f"  {len(integration_findings)} issues de integración encontrados")
    
    return {
        "local_issues": local_findings,
        "integration_issues": integration_findings,
        "total_issues": sum(len(f) for f in local_findings.values()) + len(integration_findings)
    }


# Test con dos archivos relacionados
pr_files = {
    "src/payment/processor.py": """
def process_payment(order_id: str, amount: float) -> dict:
    # Devuelve True en éxito, False en fallo
    result = payment_gateway.charge(amount)
    return result
""",
    "src/orders/checkout.py": """
from payment.processor import process_payment

def complete_checkout(order_id: str, amount: float):
    result = process_payment(order_id, amount)
    if result["status"] == "success":  # ← asume dict, pero processor devuelve True/False
        update_order_status(order_id, "completed")
"""
}

review_result = review_large_pr(pr_files)
print()
print(f"Total de issues encontrados: {review_result['total_issues']}")
print("Issues de integración:", json.dumps(review_result["integration_issues"], indent=2, ensure_ascii=False))

## Paso 4 — Message Batches API para análisis nocturno

In [ ]:
# Comparación: cuándo usar síncrona vs. batch

workflows = [
    {
        "nombre": "Verificación pre-merge",
        "api": "SÍNCRONA",
        "razón": "El dev espera el resultado para poder hacer merge. NO puede tolerar 24h de latencia.",
        "flag_ci": "claude -p \"Review PR\" --output-format json"
    },
    {
        "nombre": "Reporte nocturno de deuda técnica",
        "api": "BATCH",
        "razón": "Se genera de noche, se revisa a la mañana. Tolera 24h. Ahorra 50% en costo.",
        "flag_ci": "client.messages.batches.create()"
    },
    {
        "nombre": "Auditoría semanal de seguridad",
        "api": "BATCH",
        "razón": "Se ejecuta los domingos, resultados disponibles el lunes. No bloquea a nadie.",
        "flag_ci": "client.messages.batches.create()"
    },
    {
        "nombre": "Generación de tests para archivo modificado",
        "api": "SÍNCRONA",
        "razón": "El dev necesita los tests antes de poder commitear el PR.",
        "flag_ci": "claude -p \"Generate tests for file.py\""
    }
]

print(f"{'WORKFLOW':<35} {'API':<12} RAZÓN")
print("-" * 100)
for w in workflows:
    print(f"{w['nombre']:<35} {w['api']:<12} {w['razón']}")

print()
print("REGLA CLAVE: Batch API → sin SLA de latencia (hasta 24h).")
print("Si el flujo es BLOQUEANTE (alguien espera) → usar API síncrona.")

## Preguntas de Reflexión

1. ¿Qué flag hace que Claude Code no se cuelgue en un pipeline CI? ¿Qué hace exactamente?
2. ¿Por qué `tool_choice: {"type": "tool", "name": "report_code_review"}` es mejor que `tool_choice: "auto"` en CI?
3. Un PR de 14 archivos da resultados inconsistentes (algunos archivos con feedback detallado, otros superficial). ¿Cuál es la solución arquitectónica?
4. ¿Por qué la misma sesión de Claude que generó el código es menos efectiva para revisarlo?
5. ¿Para qué sirve el campo `detected_pattern` en los hallazgos?